# LazySlide End-to-End Pipeline
This notebook contains the complete LazySlide workflow for Whole Slide Imaging (WSI) analysis.

In [ ]:
!pip install -q lazyslide openslide-python openslide-bin huggingface_hub "scanpy[leiden]"

In [ ]:
from huggingface_hub import hf_hub_download
from wsidata import open_wsi
import lazyslide as zs
import scanpy as sc

# Download demo slide from Hugging Face
slide = hf_hub_download(
    "rendeirolab/lazyslide-data",
    "GTEX-11DXX-1626.svs",
    repo_type="dataset",
    cache_dir="."
)

wsi = open_wsi(slide)
wsi

In [ ]:
# Preprocessing: Tissue finding and tiling
zs.pl.tissue(wsi)
zs.pp.find_tissues(wsi)
zs.pp.tile_tissues(wsi, 128)
zs.pl.tiles(wsi, linewidth=0.2)

In [ ]:
# Feature extraction and analysis
# ctranspath is the stable feature extractor in the current pathology310 VM.
zs.tl.feature_extraction(wsi, "ctranspath", amp=True)

adata = wsi["ctranspath_tiles"]

sc.pp.scale(adata)
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.tl.leiden(adata, flavor="igraph", resolution=0.2)

In [ ]:
# Visualization and Export
sc.pl.umap(adata, color="leiden")
zs.pl.tiles(
    wsi,
    feature_key="ctranspath",
    color="leiden",
    alpha=0.5,
    palette=adata.uns["leiden_colors"],
    show_contours=False,
)
wsi.write()